# Pretrain & save pose VAEs (+ TCR encoders)

Pretrains the **pose VAEs** (unsupervised) and the **TCR sequence encoders**
(warm-start), and saves them to `REPO/pretrained/` so `train.ipynb` can load
instead of pretraining inline.

Two pose-corpus strategies, each with an ipTM-filter option:
* **binder + non-binder** (`which="all"`) — manifold spans both classes (default).
* **binder only** (`which="binder"`) — one-class / anomaly-style manifold.

Runtime → GPU. Data CSVs in `REPO/data/`.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import sys, os
REPO="/content/drive/MyDrive/tcrpmhc_pose_binding"
DATA_DIR=f"{REPO}/data"
PRETRAIN_DIR=f"{REPO}/pretrained"; os.makedirs(PRETRAIN_DIR,exist_ok=True)
sys.path.insert(0,f"{REPO}/src")
print("src:",os.path.isdir(f"{REPO}/src"),"| data:",os.path.isdir(DATA_DIR),"| out:",PRETRAIN_DIR)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","-r",f"{REPO}/requirements.txt"])
import json, torch
from data import pose_corpus, load_data
from pose_vae import PoseVAERaw
from train_utils import pretrain_posevae, pretrain_tcr_encoders, DEVICE
print("device:",DEVICE)

## Config

In [ ]:
PRETRAIN_EPOCHS = 60    # upper bound; early stop usually ends sooner
PATIENCE        = 8     # stop if recon loss plateaus for this many epochs    # pose VAE epochs (raise for a final run)
TCR_EPOCHS      = 15    # TCR encoder AE epochs
ROT             = ["6d","qnn"]
# corpus strategies (name -> pose_corpus kwargs)
STRATEGIES = {
 "binderNonbinder_iptm05": dict(which="all",      min_iptm=0.5),
 "binderOnly_iptm05":      dict(which="binder",   min_iptm=0.5),
 "binderNonbinder_all":    dict(which="all",      min_iptm=None),   # no ipTM filter (extra)
}

## 1. Stage 0 — TCR sequence encoders (label-agnostic, all sequences)

In [ ]:
seq = load_data(DATA_DIR, min_iptm=0.5)["seq"]         # full seq table w/ chains
TCR_ENC = pretrain_tcr_encoders(seq, epochs=TCR_EPOCHS)
torch.save(TCR_ENC, f"{PRETRAIN_DIR}/tcr_encoders.pt")
print("saved tcr_encoders.pt  (chains:", list(TCR_ENC), ")")

## 2. Stage 1 — pose VAEs per strategy × rotation encoder

In [ ]:
meta = {"trans_scale": None, "strategies": {}, "files": []}
HIST = {}
for sname, cfg in STRATEGIES.items():
    POSE_ALL, trans_scale = pose_corpus(DATA_DIR, **cfg)
    meta["trans_scale"] = trans_scale
    meta["strategies"][sname] = dict(cfg, n=int(len(POSE_ALL)))
    print(f"[{sname}] corpus n={len(POSE_ALL)}  cfg={cfg}")
    for rot in ROT:
        vae, hist = pretrain_posevae(POSE_ALL, rot, epochs=PRETRAIN_EPOCHS, return_history=True, patience=PATIENCE)
        fn = f"posevae_{sname}_{rot}.pt"
        torch.save(vae.state_dict(), f"{PRETRAIN_DIR}/{fn}")
        hist.to_csv(f"{PRETRAIN_DIR}/history_{sname}_{rot}.csv", index=False)
        HIST[(sname, rot)] = hist; meta["files"].append(fn); print("   saved", fn)
json.dump(meta, open(f"{PRETRAIN_DIR}/meta.json","w"), indent=2)
print("\nsaved meta.json | trans_scale =", round(meta["trans_scale"],3))

## Convergence diagnostics (is the epoch count enough?)
Reconstruction-loss plateau (#1), rotation/direction error in degrees (#3), and active latent units / KL-per-dim for posterior collapse (#4).

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 2, figsize=(14, 9)); ax = ax.ravel()
for (sname, rot), h in HIST.items():
    lab = f"{sname}|{rot}"
    ax[0].plot(h.epoch, h.recon, label=lab)
    ax[1].plot(h.epoch, h.kld, label=lab)
    ax[2].plot(h.epoch, h.rot_deg, label=lab); ax[2].plot(h.epoch, h.dir_deg, ls="--")
    ax[3].plot(h.epoch, h.active_units, label=lab)
ax[0].set_title("reconstruction loss (#1)"); ax[1].set_title("KL term (#4)")
ax[2].set_title("rotation (solid) & direction (dashed) recon error, deg (#3)")
ax[3].set_title("active latent units, KL/dim>0.01 (#4)")
for a in ax: a.set_xlabel("epoch"); a.legend(fontsize=6)
plt.tight_layout(); plt.savefig(f"{PRETRAIN_DIR}/pretrain_curves.png", dpi=150); plt.show()
print("convergence readout (recon slope over last 5 epochs ~0 => plateaued):")
for (sname, rot), h in HIST.items():
    last = h.recon.values[-5:]; slope = (last[-1]-last[0])/4
    print(f"  {sname}|{rot}: recon={last[-1]:.3f} slope/ep={slope:+.4f} | rot={h.rot_deg.values[-1]:.1f}deg "
          f"dir={h.dir_deg.values[-1]:.1f}deg | active={int(h.active_units.values[-1])} KL/dim={h.kl_per_dim.values[-1]:.3f}")

## 3. How to load in `train.ipynb`
```python
from pose_vae import PoseVAERaw
import torch
def load_posevae(name, rot, pretrain_dir, device):
    vae = PoseVAERaw(rotation_encoder=rot).to(device)
    vae.load_state_dict(torch.load(f"{pretrain_dir}/posevae_{name}_{rot}.pt", map_location=device))
    vae.eval(); return vae
# vae6 = load_posevae("binderNonbinder_iptm05","6d", PRETRAIN_DIR, DEVICE)
# TCR_ENC = torch.load(f"{PRETRAIN_DIR}/tcr_encoders.pt"); warm_start = make_warm_start(TCR_ENC)
```
Note: pose VAEs are pretrained on the *full* corpus once; for a strict per-fold
evaluation, pretrain within each fold's training structures instead.